# Pet segmentation: training v5 (384 px, small pets, test-time augmentation)

v5 keeps v3's model and loss (boundary ignored, CE + Dice, no class weights) and v4's clean split and cleaned trimaps,
and goes after the weak spots v3 and v4 showed:

| Weak spot | What v5 changes |
|---|---|
| Pets covering under 10% of the image score worst (v3 0.87, v4 0.84) | **384 px** input instead of 256 (2.25× the pixels on every pet) and **small-pet oversampling**: images with a pet under 10% of the frame are drawn `oversample_small`× as often, 10–30% pets 1.5× |
| Augmentation was crops, flips and colour jitter only | Adds **rotation ±15°** (rotated-in corners are ignored in the loss) and a wider crop-scale range (0.3–1.0) |
| No test-time augmentation | Every model is also scored with **TTA**: horizontal flip × 3 scales (0.75, 1, 1.25 of its training size), probabilities averaged |

Section 6b scores **v3, v3 + TTA, v4, v5, v5 + TTA** the same way on the full and the clean test set, with the size,
border and hardest-breed breakdowns, and names the best combination to plug into the ViTMatte edges notebook.
Checkpoints go to `checkpoints_v5/`; earlier models aren't touched. The clean split is reused from Drive (`clean_splits/`).

---

This notebook trains a **U-Net with an ImageNet-pretrained ResNet-34 encoder** on the Oxford-IIIT Pet trimaps, on a **GPU**.
It follows on from `EDA.ipynb`, and the main choices come from what the EDA found:

| EDA finding | Choice made here |
|---|---|
| Every image has a matching trimap of the same size, with values only {1, 2, 3} | Masks are loaded as-is and remapped to 0 background, 1 pet, 2 boundary |
| Background is most of the pixels, but only about 2:1 against the pet | Cross-entropy **plus** Dice loss, without class weights, and scale-jitter crops |
| The boundary band (class 3) is a wide region of uncertain labels | Left out of the loss and the metrics (`boundary_mode="ignore"`) |
| A few masks are empty, and some images are near-duplicates across splits | Clean split rebuilt here (section 2) and saved to Drive |
| Pets are centre-biased and often touch the image border | Random resized crops + flips so the model can't rely on position |
| Cats and dogs and the 37 breeds differ in size and edge quality | Test IoU is broken down by species, by breed and by pet size |

**Runtime:** *Runtime ▸ Change runtime type ▸ T4 GPU* (or better), then *Run all*.

**If Colab disconnects:** checkpoints are saved to Drive after every epoch (`checkpoints_v5/` in the project folder).
Choose *Run all* again and training resumes from the last finished epoch.

## 1. Setup

In [ ]:
import os, sys, random, shutil, tarfile, time, urllib.request, zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

In [ ]:
CONFIG = {
    "img_size": 384,          # must be divisible by 32 (v3/v4: 256)
    "rotate_deg": 15,         # random rotation range; rotated-in corners are ignored in the loss
    "min_crop_scale": 0.3,    # random resized crop keeps 30–100% of the image (v3/v4: 50–100%)
    "oversample_small": 4.0,  # sampling weight for images whose pet covers <10% of the frame (10–30%: 1.5)
    "batch_size": 16,
    "epochs": 40,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "boundary_mode": "ignore", # "ignore": boundary is ignore_index in loss and metrics (binary model) | "class": v1/v2 behaviour
    "dice_weight": 1.0,
    "class_weighting": "none",  # "none": plain CE + Dice | "sqrt_inv": v1's 1/sqrt(pixel frequency) weights
    "patience": 7,           # early stopping on validation mIoU
    "num_workers": 2,
    "seed": 42,
    "pretrained": True,
}
REQUIRE_GPU = True

DATA_DIR = Path("/content/dataset")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/upgaming/Animal Classification & Segmentation")
# on Colab, checkpoints live on Drive so a disconnect doesn't lose the run
CKPT_DIR = DRIVE_PROJECT_DIR / "checkpoints_v5" if IN_COLAB else Path("/content/checkpoints_v5")
CLEAN_SPLITS_DIR = DRIVE_PROJECT_DIR / "clean_splits"   # saved on Drive and reused by later runs
REBUILD_SPLITS = False      # True = recompute the clean split even if it's already on Drive
SEG_FLAG_IOU = 0.4          # mask vs pretrained segmenter IoU below this (with the segmenter seeing a pet) = flagged
SEG_MIN_PET = 0.05          # ... the segmenter must predict at least this share of the image as cat/dog
DUP_HAMMING, DUP_CORR = 4, 0.95
SIZE_BINS = [0, 0.10, 0.30, 0.50, 1.0]   # pet share of the image, for the size-grouped evaluation
SIZE_LABELS = ["<10%", "10–30%", "30–50%", ">50%"]
RESUME = True   # continue from CKPT_DIR/last.pt if it exists and was made with the same CONFIG
assert CONFIG["img_size"] % 32 == 0

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU found. In Colab choose Runtime ▸ Change runtime type ▸ T4 GPU, then run all again.")
else:
    device = torch.device("cpu")
    print("Running on CPU")
torch.backends.cudnn.benchmark = True


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG["seed"])

## 2. Data

Looks for the dataset in `/content/dataset`. If it isn't there, it copies the archives from the Drive `Data` folder when they exist,
and otherwise downloads the official Oxford-IIIT Pet archives.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

IMAGES_DIR = DATA_DIR / "images"
ANN_DIR = DATA_DIR / "annotations"
TRIMAPS_DIR = ANN_DIR / "trimaps"
ARCHIVES = {
    "images.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/images.tar.gz",
    "annotations.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/annotations.tar.gz",
}


def dataset_ready():
    return IMAGES_DIR.is_dir() and TRIMAPS_DIR.is_dir() and (ANN_DIR / "trainval.txt").exists()


def extract(archive):
    print("extracting", archive.name)
    with tarfile.open(archive) as t:
        try:
            t.extractall(DATA_DIR, filter="data")
        except TypeError:  # Python without tarfile filters
            t.extractall(DATA_DIR)


if not dataset_ready():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    drive_data = DRIVE_PROJECT_DIR / "Data"
    for name, url in ARCHIVES.items():
        local = DATA_DIR / name
        on_drive = drive_data / name
        if on_drive.exists():
            print("copying", on_drive)
            shutil.copy(on_drive, local)
        else:
            print("downloading", url)
            urllib.request.urlretrieve(url, local)
        extract(local)
        local.unlink()

assert dataset_ready(), f"dataset not found in {DATA_DIR}"
print("images:", sum(1 for _ in IMAGES_DIR.glob("*.jpg")), "| trimaps:", sum(1 for _ in TRIMAPS_DIR.glob("*.png")))

### Cleaned trimaps
Uses the cleaned trimaps from the latest annotation audit (`runs/audit_*/trimaps_clean.zip`): detached boundary strips
became background and pet specks inside the band became boundary. Falls back to the original trimaps if no audit is found.

In [ ]:
def read_list(path):
    return [l.split()[0] for l in path.read_text().splitlines() if l.strip() and not l.startswith("#")]


def breed_of(name):
    return name.rsplit("_", 1)[0]


def species_of(name):
    return "cat" if name[0].isupper() else "dog"


def files_exist(name):
    return (not name.startswith("._") and (IMAGES_DIR / f"{name}.jpg").exists()
            and (TRIMAPS_DIR / f"{name}.png").exists())


CLEAN_TRIMAPS = ANN_DIR / "trimaps_clean"
audit_zips = sorted((DRIVE_PROJECT_DIR / "runs").glob("audit_*/trimaps_clean.zip"))
if not (CLEAN_TRIMAPS.is_dir() and any(CLEAN_TRIMAPS.glob("*.png"))) and audit_zips:
    print("extracting", audit_zips[-1])
    with zipfile.ZipFile(audit_zips[-1]) as z:
        z.extractall(ANN_DIR)
LABEL_DIR = CLEAN_TRIMAPS if CLEAN_TRIMAPS.is_dir() and any(CLEAN_TRIMAPS.glob("*.png")) else TRIMAPS_DIR
print("labels from:", LABEL_DIR, "" if LABEL_DIR == CLEAN_TRIMAPS else "(no audit found: ORIGINAL trimaps)")

### Clean split (EDA section 8)
Starts from the official `trainval`/`test` lists and the same 80/20 breed-stratified train/val split as v1–v3 (seed 42), then
drops, in this order: images that don't decode, **empty masks**, masks the **pretrained segmenter** clearly disagrees with,
**near-duplicates** of an image in another split (the copy in the higher-priority split is kept: test > val > train), and
duplicates inside train/val. The result goes to `clean_splits/` on Drive with a `manifest.csv` (every image and its drop
reason) and `duplicates.csv`. If it's already on Drive it's reused.

In [ ]:
from sklearn.model_selection import train_test_split

trainval_all = [n for n in read_list(ANN_DIR / "trainval.txt") if files_exist(n)]
test_all = [n for n in read_list(ANN_DIR / "test.txt") if files_exist(n)]
_tr, _va = train_test_split(trainval_all, test_size=0.20, random_state=42, stratify=[breed_of(n) for n in trainval_all])
base_split = {**{n: "train" for n in _tr}, **{n: "val" for n in _va}, **{n: "test" for n in test_all}}


def dhash(img, size=8):
    g = np.asarray(img.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return int.from_bytes(np.packbits((g[:, 1:] > g[:, :-1]).ravel()).tobytes(), "big")


def thumb(name):
    with Image.open(IMAGES_DIR / f"{name}.jpg") as im:
        return np.asarray(im.convert("L").resize((64, 64), Image.BILINEAR), np.float32).ravel()


def build_clean_split():
    from torchvision.models.segmentation import lraspp_mobilenet_v3_large, LRASPP_MobileNet_V3_Large_Weights
    names = sorted(base_split)
    rows = []
    for n in tqdm(names, desc="decode / empty / hash"):
        r = {"filename": n, "breed": breed_of(n), "species": species_of(n), "split": base_split[n]}
        try:
            with Image.open(IMAGES_DIR / f"{n}.jpg") as im:
                im.load()
                r["dhash"] = dhash(im)
            r["img_ok"] = True
        except Exception:
            r["img_ok"], r["dhash"] = False, None
        r["empty_mask"] = not (np.array(Image.open(TRIMAPS_DIR / f"{n}.png")) == 1).any()
        rows.append(r)
    qc = pd.DataFrame(rows)

    # pretrained-segmenter screen: VOC classes cat = 8, dog = 12
    segm = lraspp_mobilenet_v3_large(weights=LRASPP_MobileNet_V3_Large_Weights.DEFAULT).to(device).eval()
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    R = 384
    seg_iou, seg_pet = {}, {}
    ok_names = qc.loc[qc.img_ok, "filename"].tolist()
    with torch.no_grad():
        for b in tqdm(range(0, len(ok_names), 16), desc="pretrained-segmenter screen"):
            batch = ok_names[b:b + 16]
            x = torch.stack([torch.from_numpy(np.array(Image.open(IMAGES_DIR / f"{n}.jpg").convert("RGB")
                                                        .resize((R, R), Image.BILINEAR))).permute(2, 0, 1) for n in batch])
            x = (x.float().div(255).to(device) - mean) / std
            pred = segm(x)["out"].argmax(1)
            pet = ((pred == 8) | (pred == 12)).cpu().numpy()
            for n, pp in zip(batch, pet):
                t = np.array(Image.open(TRIMAPS_DIR / f"{n}.png").resize((R, R), Image.NEAREST))
                gt = t != 2
                u = (pp | gt).sum()
                seg_iou[n] = (pp & gt).sum() / u if u else 1.0
                seg_pet[n] = pp.mean()
    qc["seg_iou"], qc["seg_pet_share"] = qc.filename.map(seg_iou), qc.filename.map(seg_pet)
    qc["seg_flag"] = (qc.seg_iou < SEG_FLAG_IOU) & (qc.seg_pet_share >= SEG_MIN_PET)

    # near-duplicates: dHash Hamming <= DUP_HAMMING, confirmed by thumbnail correlation
    hashed = qc[qc.img_ok].reset_index(drop=True)
    H = np.array([h for h in hashed.dhash], dtype=np.uint64)
    POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    cand = []
    for i0 in range(0, len(H), 512):
        x = (H[i0:i0 + 512, None] ^ H[None, :]).view(np.uint8).reshape(-1, len(H), 8)
        d = POP[x].sum(-1)
        ii, jj = np.nonzero(d <= DUP_HAMMING)
        cand += [(i0 + i, j, int(d[i, j])) for i, j in zip(ii, jj) if i0 + i < j]
    dup_rows = []
    for i, j, dist in tqdm(cand, desc="confirming duplicates"):
        a, b = hashed.filename[i], hashed.filename[j]
        corr = float(np.corrcoef(thumb(a), thumb(b))[0, 1])
        if corr > DUP_CORR:
            dup_rows.append({"a": a, "split_a": base_split[a], "b": b, "split_b": base_split[b],
                             "hamming": dist, "thumb_corr": round(corr, 4)})
    dups = pd.DataFrame(dup_rows, columns=["a", "split_a", "b", "split_b", "hamming", "thumb_corr"])

    qc["drop_reason"] = ""
    def mark(sel, reason):
        qc.loc[sel & (qc.drop_reason == ""), "drop_reason"] = reason
    mark(~qc.img_ok, "image does not decode")
    mark(qc.empty_mask, "empty mask")
    mark(qc.seg_flag, "disagrees with pretrained segmenter")
    rank = {"test": 0, "val": 1, "train": 2}
    leak, twin = set(), set()
    for d in dups.itertuples():
        if d.split_a != d.split_b:
            leak.add(d.a if rank[d.split_a] > rank[d.split_b] else d.b)
        else:
            twin.add(d.b)
    mark(qc.filename.isin(leak), "duplicate of an image in another split")
    mark(qc.filename.isin(twin) & (qc.split != "test"), "duplicate inside the same split")
    qc["keep"] = qc.drop_reason == ""

    CLEAN_SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    for s in ["train", "val", "test"]:
        qc.loc[qc.keep & (qc.split == s), "filename"].to_csv(CLEAN_SPLITS_DIR / f"{s}.txt", index=False, header=False)
    qc.drop(columns=["dhash"]).to_csv(CLEAN_SPLITS_DIR / "manifest.csv", index=False)
    dups.to_csv(CLEAN_SPLITS_DIR / "duplicates.csv", index=False)
    return qc.drop(columns=["dhash"]), dups


if REBUILD_SPLITS or not all((CLEAN_SPLITS_DIR / f).exists() for f in ["train.txt", "val.txt", "test.txt", "manifest.csv", "duplicates.csv"]):
    manifest, dups = build_clean_split()
    print("clean split built and saved to", CLEAN_SPLITS_DIR)
else:
    manifest = pd.read_csv(CLEAN_SPLITS_DIR / "manifest.csv").fillna({"drop_reason": ""})
    dups = pd.read_csv(CLEAN_SPLITS_DIR / "duplicates.csv")
    print("reusing the clean split from", CLEAN_SPLITS_DIR)

splits = {s: [n for n in read_list(CLEAN_SPLITS_DIR / f"{s}.txt") if files_exist(n)] for s in ("train", "val", "test")}
full_test = manifest.query("split == 'test' and drop_reason not in ['empty mask', 'image does not decode']").filename.tolist()
# test images whose near-twin sat in train/val (v1–v3 trained with that twin)
leaked_test = sorted({d.a if d.split_a == "test" else d.b for d in dups.itertuples()
                      if (d.split_a == "test") != (d.split_b == "test")})

print("\ndropped:")
display(manifest[manifest.drop_reason != ""].groupby(["drop_reason", "split"]).size().unstack(fill_value=0))
for s, v in splits.items():
    print(f"  clean {s:5s}: {len(v):5d} images, {len({breed_of(n) for n in v})} breeds")
print(f"  full test (only empty masks dropped, as v1–v3): {len(full_test)} | test images with a twin in train/val: {len(leaked_test)}")

In [ ]:
def quick_overlay(img, t, alpha=0.45):
    out = img.astype(np.float32).copy()
    for v, col in [(1, (228, 26, 28)), (3, (255, 217, 47))]:
        m = t == v
        out[m] = (1 - alpha) * out[m] + alpha * np.array(col)
    return out.astype(np.uint8)


def gallery(names, title, k=6, extra=None):
    names = list(names)[:k]
    if not names:
        print("no examples for:", title)
        return
    fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4.4), squeeze=False)
    for ax, n in zip(axes[0], names):
        ax.imshow(quick_overlay(np.array(Image.open(IMAGES_DIR / f"{n}.jpg").convert("RGB")),
                                np.array(Image.open(TRIMAPS_DIR / f"{n}.png"))))
        ax.set_title(n + (f"\n{extra(n)}" if extra else ""), fontsize=9)
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


seg_iou_of = manifest.set_index("filename").seg_iou
flagged = manifest[manifest.drop_reason == "disagrees with pretrained segmenter"].sort_values("seg_iou")
gallery(flagged.filename, "Dropped: mask disagrees with the pretrained segmenter (red = pet, yellow = boundary)",
        extra=lambda n: f"segmenter IoU {seg_iou_of[n]:.2f}")
cross = dups[dups.split_a != dups.split_b].head(3)
for d in cross.itertuples():
    gallery([d.a, d.b], f"Near-duplicate across splits: {d.split_a} vs {d.split_b} (thumbnail corr {d.thumb_corr:.3f})", k=2)

### Labels
Trimap values are 1 = pet, 2 = background, 3 = boundary. They are remapped to 0 = background, 1 = pet, and either 2 = boundary
(`boundary_mode="class"`) or 255 = ignored (`boundary_mode="ignore"`).

In [ ]:
IGNORE = 255
if CONFIG["boundary_mode"] == "class":
    CLASS_NAMES = ["background", "pet", "boundary"]
    TRIMAP_TO_LABEL = {1: 1, 2: 0, 3: 2}
elif CONFIG["boundary_mode"] == "ignore":
    CLASS_NAMES = ["background", "pet"]
    TRIMAP_TO_LABEL = {1: 1, 2: 0, 3: IGNORE}
else:
    raise ValueError(CONFIG["boundary_mode"])
NUM_CLASSES = len(CLASS_NAMES)

LUT = np.full(256, IGNORE, dtype=np.uint8)
for k, v in TRIMAP_TO_LABEL.items():
    LUT[k] = v


def load_label(name):
    return LUT[np.array(Image.open(LABEL_DIR / f"{name}.png"))]


def load_rgb(name):
    return np.array(Image.open(IMAGES_DIR / f"{name}.jpg").convert("RGB"))

### Dataset and augmentation

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CONFIG["img_size"]

# torchvision v2 transforms apply the same geometry to image and mask (nearest-neighbour for masks);
# colour jitter and normalisation only touch the image.
train_tf = v2.Compose([
    v2.RandomRotation(CONFIG["rotate_deg"], fill={tv_tensors.Image: 0, tv_tensors.Mask: IGNORE}),
    v2.RandomResizedCrop((S, S), scale=(CONFIG["min_crop_scale"], 1.0), ratio=(0.75, 1.33), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.RandomApply([v2.ColorJitter(0.3, 0.3, 0.3, 0.05)], p=0.8),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
eval_tf = v2.Compose([
    v2.Resize((S, S), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])


class PetSegDataset(Dataset):
    def __init__(self, names, transform):
        self.names, self.transform = list(names), transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        img = tv_tensors.Image(torch.from_numpy(load_rgb(name)).permute(2, 0, 1))
        mask = tv_tensors.Mask(torch.from_numpy(load_label(name)))
        img, mask = self.transform(img, mask)
        return img, mask.long(), i


datasets = {
    "train": PetSegDataset(splits["train"], train_tf),
    "val": PetSegDataset(splits["val"], eval_tf),
    "test": PetSegDataset(splits["test"], eval_tf),
}
loader_kw = dict(batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"],
                 pin_memory=device.type == "cuda", persistent_workers=CONFIG["num_workers"] > 0)


def animal_share(name):
    t = np.array(Image.open(LABEL_DIR / f"{name}.png"))
    return (t != 2).mean()


train_shares = np.array([animal_share(n) for n in tqdm(splits["train"], desc="pet shares for sampling")])
sample_w = np.where(train_shares < 0.10, CONFIG["oversample_small"], np.where(train_shares < 0.30, 1.5, 1.0))
sampler = WeightedRandomSampler(torch.tensor(sample_w, dtype=torch.double), num_samples=len(sample_w), replacement=True)
print(f"small pets (<10%): {(train_shares < 0.10).sum()} images, drawn {CONFIG['oversample_small']}x; "
      f"expected share per epoch {sample_w[train_shares < 0.10].sum() / sample_w.sum():.1%} "
      f"(was {(train_shares < 0.10).mean():.1%})")

loaders = {
    "train": DataLoader(datasets["train"], sampler=sampler, drop_last=True, **loader_kw),
    "val": DataLoader(datasets["val"], shuffle=False, **loader_kw),
    "test": DataLoader(datasets["test"], shuffle=False, **loader_kw),
}

In [ ]:
def denormalize(x):
    return (x.permute(1, 2, 0).cpu().numpy() * STD + MEAN).clip(0, 1)


def label_to_rgb(m):
    palette = np.array([[40, 40, 40], [228, 26, 28], [255, 217, 47]], dtype=np.uint8)
    m = np.asarray(m)
    out = np.full((*m.shape, 3), 255, dtype=np.uint8)  # white = ignored
    valid = m != IGNORE
    out[valid] = palette[m[valid]]
    return out


imgs, masks, _ = next(iter(loaders["train"]))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k in range(4):
    axes[0, k].imshow(denormalize(imgs[k]))
    axes[1, k].imshow(label_to_rgb(masks[k]))
    for ax in axes[:, k]:
        ax.axis("off")
fig.suptitle("Augmented training samples (dark = background, red = pet, yellow = boundary)")
plt.tight_layout()
plt.show()

### Foreground distribution and class balance
Exact numbers on the training masks at full resolution. The trimap is read directly, so the shares are the same in both
`boundary_mode`s: **pet share** is pet + boundary (the whole animal) as a fraction of the image.

In [ ]:
trimap_px = np.zeros(4)   # index 1 pet, 2 background, 3 boundary
pet_shares, bnd_rows = [], []
for name in tqdm(splits["train"], desc="pixel counts"):
    t = np.array(Image.open(LABEL_DIR / f"{name}.png"))
    c = np.bincount(t.ravel(), minlength=4)[:4]
    trimap_px += c
    pet_shares.append((c[1] + c[3]) / t.size)
    bnd_rows.append({"breed": breed_of(name), "boundary_share": c[3] / t.size,
                     "boundary_vs_pet": c[3] / c[1] if c[1] else np.nan})
pet_shares = pd.Series(pet_shares, index=splits["train"], name="pet_share")

share = trimap_px / trimap_px[1:].sum()
display(pd.DataFrame({"pixel_share": [share[2], share[1], share[3]]},
                     index=["background", "pet", "boundary"]).round(4))
print(f"background : pet = {share[2] / share[1]:.2f} : 1   (background : pet+boundary = {share[2] / (share[1] + share[3]):.2f} : 1)")
print(f"pet share: mean {pet_shares.mean():.1%}, median {pet_shares.median():.1%}, "
      f"10–50%: {pet_shares.between(0.10, 0.50).mean():.1%} of images, max {pet_shares.max():.1%}")
print(f"at {S}x{S} input, a pet covering 5% of the image is about {0.05 * S * S:,.0f} px")

size_bin_counts = pd.cut(pet_shares, SIZE_BINS, labels=SIZE_LABELS, include_lowest=True).value_counts().reindex(SIZE_LABELS)
display(size_bin_counts.to_frame("train images"))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pet_shares, bins=50, color="#4c72b0")
for v, lbl, ls in [(pet_shares.mean(), "mean", "--"), (pet_shares.median(), "median", "-")]:
    ax.axvline(v, color="k", linestyle=ls, label=f"{lbl} {v:.1%}")
for b in SIZE_BINS[1:-1]:
    ax.axvline(b, color="#c44e52", alpha=0.4)
ax.set_xlabel("pet share of the image (pet + boundary)")
ax.set_ylabel("training images")
ax.set_title("Foreground distribution (red lines: size-bin edges used in evaluation)")
ax.legend()
plt.tight_layout()
plt.show()

### How big is the boundary band?
Per-breed boundary share of the image, and boundary area relative to the pet-interior area (class 3 / class 1).
Merging the band into the pet would grow the target by roughly the `boundary_vs_pet` ratio.

In [ ]:
bnd = pd.DataFrame(bnd_rows)
bnd_breed = bnd.groupby("breed")[["boundary_share", "boundary_vs_pet"]].mean().sort_values("boundary_share")
print(f"boundary share of the image by breed: {bnd_breed.boundary_share.min():.1%} – {bnd_breed.boundary_share.max():.1%} "
      f"(overall {share[3]:.1%})")
print(f"boundary area / pet-interior area: {share[3] / share[1]:.0%} overall, "
      f"{bnd_breed.boundary_vs_pet.min():.0%} – {bnd_breed.boundary_vs_pet.max():.0%} by breed")
display(pd.concat([bnd_breed.head(3), bnd_breed.tail(3)]).round(3))

### Class weights
Background vs pet is only about 2:1, so by default the loss is plain cross-entropy + Dice with **no class weights**
(Dice already balances the classes). Set `class_weighting="sqrt_inv"` to use v1's `1/sqrt(frequency)` weights instead.

In [ ]:
counts = np.zeros(NUM_CLASSES)
for k, v in TRIMAP_TO_LABEL.items():
    if v != IGNORE:
        counts[v] += trimap_px[k]
freq = counts / counts.sum()
if CONFIG["class_weighting"] == "sqrt_inv":
    class_weights = 1 / np.sqrt(freq)
    class_weights /= class_weights.mean()
elif CONFIG["class_weighting"] == "none":
    class_weights = np.ones(NUM_CLASSES)
else:
    raise ValueError(CONFIG["class_weighting"])
display(pd.DataFrame({"pixel_share": freq.round(4), "weight": class_weights.round(3)}, index=CLASS_NAMES))

## 3. Model: U-Net with a ResNet-34 encoder

In [ ]:
def conv_bn_relu(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class DecoderBlock(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.block = nn.Sequential(conv_bn_relu(c_in + c_skip, c_out), conv_bn_relu(c_out, c_out))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNetResNet34(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        r = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)          # 1/2,  64 ch
        self.pool = r.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = r.layer1, r.layer2, r.layer3, r.layer4  # 1/4 .. 1/32
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.head = nn.Conv2d(16, num_classes, 1)

    def forward(self, x):
        s0 = self.stem(x)
        e1 = self.enc1(self.pool(s0))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d = self.dec4(e4, e3)
        d = self.dec3(d, e2)
        d = self.dec2(d, e1)
        d = self.dec1(d, s0)
        return self.head(self.dec0(d))


model = UNetResNet34(NUM_CLASSES, CONFIG["pretrained"]).to(device).to(memory_format=torch.channels_last)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## 4. Loss and metrics

In [ ]:
class CEDiceLoss(nn.Module):
    """Class-weighted cross-entropy + soft Dice; pixels labelled IGNORE count for neither."""

    def __init__(self, weight, dice_weight=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight, ignore_index=IGNORE)
        self.dice_weight = dice_weight

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        valid = (target != IGNORE).unsqueeze(1)
        prob = logits.float().softmax(dim=1) * valid
        onehot = F.one_hot(torch.where(target == IGNORE, 0, target), logits.shape[1]).permute(0, 3, 1, 2) * valid
        inter = (prob * onehot).sum(dim=(0, 2, 3))
        denom = prob.sum(dim=(0, 2, 3)) + onehot.sum(dim=(0, 2, 3))
        dice = 1 - ((2 * inter + 1) / (denom + 1)).mean()
        return ce + self.dice_weight * dice


def confusion(pred, target, n=NUM_CLASSES):
    k = target != IGNORE
    return torch.bincount(target[k] * n + pred[k], minlength=n * n).reshape(n, n)


def scores(cm):
    cm = cm.double()
    tp = cm.diag()
    iou = tp / (cm.sum(0) + cm.sum(1) - tp).clamp(min=1)
    dice = 2 * tp / (cm.sum(0) + cm.sum(1)).clamp(min=1)
    return {"miou": iou.mean().item(), "pixel_acc": (tp.sum() / cm.sum().clamp(min=1)).item(),
            **{f"iou_{c}": iou[i].item() for i, c in enumerate(CLASS_NAMES)},
            **{f"dice_{c}": dice[i].item() for i, c in enumerate(CLASS_NAMES)}}


def _pet_masks(pred, target):
    valid = target != IGNORE
    return (pred != 0) & valid, (target != 0) & valid


def pet_iou(pred, target):
    """Per-image pet-vs-background IoU over non-ignored pixels (with boundary ignored: non-boundary pixels only)."""
    p, t = _pet_masks(pred, target)
    inter = (p & t).flatten(1).sum(1).float()
    union = (p | t).flatten(1).sum(1).float()
    return torch.where(union > 0, inter / union.clamp(min=1), torch.ones_like(union))


def pet_dice(pred, target):
    """Per-image pet Dice over the same pixels as pet_iou."""
    p, t = _pet_masks(pred, target)
    inter = (p & t).flatten(1).sum(1).float()
    total = (p.flatten(1).sum(1) + t.flatten(1).sum(1)).float()
    return torch.where(total > 0, 2 * inter / total.clamp(min=1), torch.ones_like(total))


criterion = CEDiceLoss(torch.tensor(class_weights, dtype=torch.float32, device=device), CONFIG["dice_weight"])

## 5. Training

In [ ]:
use_amp = device.type == "cuda"
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CONFIG["lr"], pct_start=0.1,
                                                total_steps=CONFIG["epochs"] * len(loaders["train"]))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_epoch(loader, train):
    model.train(train)
    total_loss, n_seen = 0.0, 0
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=device)
    pet_ious = []
    with torch.set_grad_enabled(train):
        for x, y, _ in tqdm(loader, leave=False, desc="train" if train else "eval"):
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            pred = logits.argmax(1)
            cm += confusion(pred, y)
            pet_ious.append(pet_iou(pred, y))
            total_loss += loss.item() * len(x)
            n_seen += len(x)
    return {"loss": total_loss / n_seen, **scores(cm), "pet_iou": torch.cat(pet_ious).mean().item()}


CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_path, last_path = CKPT_DIR / "best.pt", CKPT_DIR / "last.pt"
history, best_miou, bad_epochs, start_epoch = [], -1.0, 0, 1

if RESUME and last_path.exists():
    last = torch.load(last_path, map_location=device, weights_only=False)
    if last["config"] == CONFIG:
        model.load_state_dict(last["model"])
        optimizer.load_state_dict(last["optimizer"])
        scheduler.load_state_dict(last["scheduler"])
        scaler.load_state_dict(last["scaler"])
        history, best_miou, bad_epochs = last["history"], last["best_miou"], last["bad_epochs"]
        start_epoch = last["epoch"] + 1
        print(f"resuming after epoch {last['epoch']} (best val mIoU so far {best_miou:.4f})")
    else:
        print("last.pt was made with a different CONFIG, starting from scratch")
    del last

stopped_early = bool(history) and bad_epochs >= CONFIG["patience"]
for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    if stopped_early:
        break
    t0 = time.time()
    tr = run_epoch(loaders["train"], train=True)
    va = run_epoch(loaders["val"], train=False)
    history.append({"epoch": epoch, "lr": scheduler.get_last_lr()[0],
                    **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}})
    improved = va["miou"] > best_miou
    if improved:
        best_miou, bad_epochs = va["miou"], 0
        torch.save({"model": model.state_dict(), "config": CONFIG, "class_names": CLASS_NAMES, "epoch": epoch}, best_path)
    else:
        bad_epochs += 1
    print(f"epoch {epoch:3d} | {time.time() - t0:5.0f}s | train loss {tr['loss']:.4f} mIoU {tr['miou']:.4f} | "
          f"val loss {va['loss']:.4f} mIoU {va['miou']:.4f} pet IoU {va['pet_iou']:.4f}" + ("  *best*" if improved else ""))
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "history": history, "best_miou": best_miou, "bad_epochs": bad_epochs,
                "epoch": epoch, "config": CONFIG}, last_path)
    if bad_epochs >= CONFIG["patience"]:
        print(f"early stopping: no val mIoU improvement for {CONFIG['patience']} epochs")
        stopped_early = True

hist = pd.DataFrame(history)
hist.to_csv(CKPT_DIR / "history.csv", index=False)
print(f"best val mIoU {best_miou:.4f} -> {best_path}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(hist.epoch, hist.train_loss, label="train")
axes[0].plot(hist.epoch, hist.val_loss, label="val")
axes[0].set_title("loss")
axes[1].plot(hist.epoch, hist.train_miou, label="train")
axes[1].plot(hist.epoch, hist.val_miou, label="val")
axes[1].set_title("mIoU")
for c in CLASS_NAMES:
    axes[2].plot(hist.epoch, hist[f"val_iou_{c}"], label=c)
axes[2].plot(hist.epoch, hist.val_pet_iou, "k--", label="pet (whole animal)")
axes[2].set_title("validation IoU per class")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Test-set evaluation
Uses the best checkpoint. Metrics are computed at the training resolution (`img_size`). With `boundary_mode="ignore"`,
**pet IoU / pet Dice** compare pet vs background **on non-boundary pixels only**: the same pixels the loss trained on.
`iou_pet` / `dice_pet` are computed over the whole test set, while `pet_iou` / `pet_dice` are per-image means.

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=device)
rows = []
with torch.no_grad():
    for x, y, idx in tqdm(loaders["test"], desc="test"):
        x = x.to(device).to(memory_format=torch.channels_last)
        y = y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x).argmax(1)
        cm += confusion(pred, y)
        shares = (y != 0).flatten(1).float().mean(1)   # pet + boundary (IGNORE counts as boundary)
        for i, piou, pdice, sh in zip(idx.tolist(), pet_iou(pred, y).tolist(), pet_dice(pred, y).tolist(), shares.tolist()):
            name = datasets["test"].names[i]
            rows.append({"filename": name, "breed": breed_of(name), "species": species_of(name),
                         "pet_share": sh, "pet_iou": piou, "pet_dice": pdice})

test_scores = scores(cm)
per_image = pd.DataFrame(rows)
test_scores["pet_iou"] = per_image.pet_iou.mean()
test_scores["pet_dice"] = per_image.pet_dice.mean()
print(f"best epoch: {ckpt['epoch']}")
display(pd.Series(test_scores, name="test").round(4).to_frame())

In [ ]:
display(per_image.groupby("species").pet_iou.agg(["mean", "median", "count"]).round(4))

breed_iou = per_image.groupby(["species", "breed"]).pet_iou.mean().reset_index().sort_values("pet_iou")
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(breed_iou.breed, breed_iou.pet_iou, color=breed_iou.species.map({"cat": "#1f77b4", "dog": "#ff7f0e"}))
ax.set_ylim(max(0, breed_iou.pet_iou.min() - 0.05), 1)
ax.set_ylabel("mean pet IoU")
ax.set_title("Test pet IoU by breed (blue = cat, orange = dog)")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()
print("hardest breeds:", ", ".join(breed_iou.head(5).breed))

### IoU by pet size
Test images grouped by how much of the image the pet covers. Small pets (<10%) are expected to score worst.

In [ ]:
per_image["size_bin"] = pd.cut(per_image.pet_share, SIZE_BINS, labels=SIZE_LABELS, include_lowest=True)
by_size = (per_image.groupby("size_bin", observed=False)
           .agg(images=("pet_iou", "size"), mean_pet_share=("pet_share", "mean"),
                mean_pet_iou=("pet_iou", "mean"), median_pet_iou=("pet_iou", "median"))
           .reindex(SIZE_LABELS))
display(by_size.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(by_size.index.astype(str), by_size.mean_pet_iou, color="#4c72b0")
for b, n in zip(bars, by_size.images):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"n={n}", ha="center", va="bottom")
ax.set_ylim(max(0, by_size.mean_pet_iou.min() - 0.1), 1)
ax.set_xlabel("pet share of the image")
ax.set_ylabel("mean pet IoU")
ax.set_title("Test pet IoU by pet size")
plt.tight_layout()
plt.show()

In [ ]:
def show_predictions(names, title):
    fig, axes = plt.subplots(len(names), 3, figsize=(11, 3.4 * len(names)), squeeze=False)
    ds = PetSegDataset(names, eval_tf)
    with torch.no_grad():
        for row, (x, y, i) in zip(axes, ds):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                pred = model(x[None].to(device).to(memory_format=torch.channels_last)).argmax(1)[0].cpu()
            piou = pet_iou(pred[None], y[None]).item()
            row[0].imshow(denormalize(x))
            row[1].imshow(label_to_rgb(y))
            row[2].imshow(label_to_rgb(pred))
            row[0].set_title(ds.names[i])
            row[1].set_title("ground truth")
            row[2].set_title(f"prediction (pet IoU {piou:.2f})")
            for ax in row:
                ax.axis("off")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


ranked = per_image.sort_values("pet_iou")
show_predictions(ranked.head(4).filename.tolist(), "Worst test predictions")
show_predictions(ranked.tail(4).filename.tolist()[::-1], "Best test predictions")

## 6b. v3, v3 + TTA, v4, v5, v5 + TTA
Every model is scored the same way: pet vs background, boundary excluded, cleaned trimaps, labels at 384 px. Each model
sees the input at **its own training size** (v3/v4: 256, v5: 384) and its prediction is resized to the label grid.

**TTA** = horizontal flip × input scales 0.75 / 1 / 1.25 of the model's training size; softmax probabilities averaged.
Two test sets: **full** (official test, only empty masks dropped: the honest headline) and **clean** (v4's clean split).

In [ ]:
TTA_SCALES = [0.75, 1.0, 1.25]


def scores_binary(cm2):
    cm2 = cm2.double()
    tp = cm2.diag()
    iou = tp / (cm2.sum(0) + cm2.sum(1) - tp).clamp(min=1)
    dice = 2 * tp / (cm2.sum(0) + cm2.sum(1)).clamp(min=1)
    return {"iou_pet (dataset)": iou[1].item(), "dice_pet (dataset)": dice[1].item(),
            "iou_background": iou[0].item(), "miou (2-class)": iou.mean().item(),
            "pixel_acc": (tp.sum() / cm2.sum().clamp(min=1)).item()}


@torch.no_grad()
def pet_prob(net, x, native, tta):
    h, w = x.shape[-2:]
    acc, n = 0, 0
    for s in (TTA_SCALES if tta else [1.0]):
        sz = max(64, int(round(native * s / 32)) * 32)
        xi = F.interpolate(x, size=(sz, sz), mode="bilinear", align_corners=False, antialias=True)
        for flip in ((False, True) if tta else (False,)):
            xin = (xi.flip(-1) if flip else xi).contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                lo = net(xin)
            lo = lo.float().flip(-1) if flip else lo.float()
            acc = acc + F.interpolate(lo, size=(h, w), mode="bilinear", align_corners=False).softmax(1)[:, 1]
            n += 1
    return acc / n


def eval_on(net, native, names, tta):
    net.eval()
    ds = PetSegDataset(names, eval_tf)
    dl = DataLoader(ds, shuffle=False, batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"])
    cm2 = torch.zeros(2, 2, dtype=torch.long, device=device)
    rows = []
    for x, y, idx in tqdm(dl, leave=False, desc=f"eval{' +TTA' if tta else ''} ({len(names)} images)"):
        x, y = x.to(device), y.to(device)
        p = (pet_prob(net, x, native, tta) > 0.5).long()
        cm2 += confusion(p, y, n=2)
        animal = y != 0
        border = animal[:, 0].any(1) | animal[:, -1].any(1) | animal[:, :, 0].any(1) | animal[:, :, -1].any(1)
        for i, piou, pdice, sh, bd in zip(idx.tolist(), pet_iou(p, y).tolist(), pet_dice(p, y).tolist(),
                                           animal.flatten(1).float().mean(1).tolist(), border.tolist()):
            n = ds.names[i]
            rows.append({"filename": n, "breed": breed_of(n), "species": species_of(n), "pet_iou": piou,
                         "pet_dice": pdice, "pet_share": sh, "touches_border": bd})
    return cm2, pd.DataFrame(rows)


def summarize(cm2, df):
    s = scores_binary(cm2)
    s["pet_iou (per-image mean)"] = df.pet_iou.mean()
    s["pet_dice (per-image mean)"] = df.pet_dice.mean()
    sizes = pd.cut(df.pet_share, SIZE_BINS, labels=SIZE_LABELS, include_lowest=True)
    for lbl, v in df.pet_iou.groupby(sizes, observed=False).mean().items():
        s[f"pet_iou {lbl}"] = v
    s["pet_iou touches border"] = df.loc[df.touches_border, "pet_iou"].mean()
    s["pet_iou not touching"] = df.loc[~df.touches_border, "pet_iou"].mean()
    s["images"] = len(df)
    return s


def load_model(folder):
    path = DRIVE_PROJECT_DIR / folder / "best.pt"
    if not path.exists():
        print("not found, skipped:", path)
        return None
    ck = torch.load(path, map_location=device, weights_only=False)
    assert len(ck["class_names"]) == 2, f"{folder} is not a boundary-ignored (2-class) model"
    net = UNetResNet34(2, pretrained=False).to(device).to(memory_format=torch.channels_last)
    net.load_state_dict(ck["model"])
    return net, ck["config"]["img_size"]


candidates = {"v5": (model, S)}
for name, folder in [("v3", "checkpoints_v3"), ("v4", "checkpoints_v4")]:
    m = load_model(folder)
    if m:
        candidates[name] = m
runs = [("v3", False), ("v3", True), ("v4", False), ("v5", False), ("v5", True)]
runs = [(m, t) for m, t in runs if m in candidates]

tables, per_image_all = {}, []
for set_name, names in {"full test": full_test, "clean test": splits["test"]}.items():
    cols = {}
    for m, tta in runs:
        label = m + (" + TTA" if tta else "")
        cm2, df = eval_on(*candidates[m], names, tta)
        cols[label] = summarize(cm2, df)
        per_image_all.append(df.assign(model=label, test_set=set_name))
    tables[set_name] = pd.DataFrame(cols)
    print(f"\n=== {set_name} ===")
    display(tables[set_name].round(4))
tables["full test"].to_csv(CKPT_DIR / "comparison_full_test.csv")
tables["clean test"].to_csv(CKPT_DIR / "comparison_clean_test.csv")
per_image_all = pd.concat(per_image_all, ignore_index=True)
per_image_all.to_csv(CKPT_DIR / "per_image_all_models.csv", index=False)

full = tables["full test"]
best = full.loc["iou_pet (dataset)"].idxmax()
print(f"\nbest on the full test (pet IoU, whole set): {best} = {full.loc['iou_pet (dataset)', best]:.4f}  |  "
      f"small pets: {full.loc['pet_iou <10%', best]:.4f}")

In [ ]:
pf = per_image_all.query("test_set == 'full test'")
breed_tbl = pf.pivot_table(index="breed", columns="model", values="pet_iou", aggfunc="mean")
hardest = breed_tbl.sort_values(breed_tbl.columns[0]).head(8)
print("Hardest breeds on the full test (by v3), mean pet IoU per model:")
display(hardest[[c for c in ["v3", "v3 + TTA", "v4", "v5", "v5 + TTA"] if c in hardest.columns]].round(4))

fig, ax = plt.subplots(figsize=(9, 4))
size_rows = [f"pet_iou {l}" for l in SIZE_LABELS]
full.loc[size_rows].rename(index=lambda r: r.replace("pet_iou ", "")).plot.bar(ax=ax, rot=0)
ax.set_ylim(max(0, full.loc[size_rows].min().min() - 0.05), 1)
ax.set_ylabel("mean pet IoU")
ax.set_xlabel("pet share of the image")
ax.set_title("Full test: pet IoU by pet size")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Save results to Drive

In [ ]:
per_image.to_csv(CKPT_DIR / "test_per_image.csv", index=False)
pd.Series(test_scores).to_csv(CKPT_DIR / "test_scores.csv")

if IN_COLAB and DRIVE_PROJECT_DIR.exists():
    run_dir = DRIVE_PROJECT_DIR / "runs" / datetime.now().strftime("v5_%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    for f in ["best.pt", "history.csv", "test_per_image.csv", "test_scores.csv", "comparison_full_test.csv", "comparison_clean_test.csv", "per_image_all_models.csv"]:
        shutil.copy(CKPT_DIR / f, run_dir / f)
    print("saved to", run_dir)
else:
    print("results kept in", CKPT_DIR)